# Part 2 — LeViT predictions (run in a FRESH Colab runtime)
LeViT needs `TF_USE_LEGACY_KERAS=1`, which conflicts with Swin/ViT in the same
session — so this runs alone, same as earlier in this project. Run this in a
**fresh runtime** (Runtime → Disconnect and delete runtime, then reconnect).

## Cell 1 — Setup

In [1]:
!pip install -U keras-cv-attention-models tf_keras -q

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import shutil
import zipfile
import json as jsonlib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from keras_cv_attention_models import levit
from sklearn.metrics import accuracy_score
import logging
tf.get_logger().setLevel('ERROR')
logging.getLogger('tensorflow').setLevel(logging.ERROR)

print("All imports successful!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.1/191.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.21.0 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.21.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0

## Cell 2 — Drive mount

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3 — Restore test set

In [3]:
split_zip_path = '/content/drive/MyDrive/split_dataset.zip'
csv_zip_path = '/content/drive/MyDrive/thesis_dataset_csv-20260716T043638Z-1-001.zip'

if not os.path.exists('/content/test'):
    shutil.copy(split_zip_path, '/content/split_dataset.zip')
    with zipfile.ZipFile('/content/split_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/')

if not os.path.exists('/content/csv_data'):
    with zipfile.ZipFile(csv_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/csv_data')

print("Test folder ready:", os.path.exists('/content/test'))

Test folder ready: True


## Cell 4 — Load test CSV + config + dataset

In [4]:
test_df = pd.read_csv('/content/csv_data/thesis_dataset_csv/test_data.csv')
test_df['filepath'] = test_df['filepath'].str.replace('/content/split_dataset', '/content')

class_names = sorted(test_df['label'].unique())
num_classes = len(class_names)
label_to_index = {name: i for i, name in enumerate(class_names)}
true_labels = test_df['label'].map(label_to_index).values

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
filepaths = test_df['filepath'].values

def load_float32(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img

def make_ds(load_fn):
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds_float32 = make_ds(load_float32)
print("Test samples:", len(test_df))

Test samples: 2538


## Cell 5 — Load LeViT-128, predict, sanity check

In [5]:
levit_backbone = levit.LeViT128(input_shape=(224, 224, 3), num_classes=0, pretrained="imagenet")
levit_backbone.trainable = False
levit_inputs = layers.Input(shape=(224, 224, 3), dtype='float32')

class TorchNormalize(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.mean = tf.constant([0.485, 0.456, 0.406]) * 255.0
        self.std = tf.constant([0.229, 0.224, 0.225]) * 255.0
    def call(self, x):
        return (x - self.mean) / self.std

x = TorchNormalize()(levit_inputs)
x = levit_backbone(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
levit_outputs = layers.Dense(num_classes, activation='softmax')(x)
levit_model = models.Model(levit_inputs, levit_outputs)
levit_model.load_weights('/content/drive/MyDrive/thesis_levit_outputs/best_levit_model.keras')
print("LeViT loaded")

pred_levit = levit_model.predict(test_ds_float32, verbose=1)
acc = accuracy_score(true_labels, np.argmax(pred_levit, axis=1))
print(f"LeViT test accuracy: {acc:.4f}")

37854624/37854624 [==============================] - 0s 0us/step
>>>> Load pretrained from: /root/.keras/models/levit128_imagenet.h5
LeViT loaded
80/80 [==============================] - 16s 144ms/step
LeViT test accuracy: 0.9515


## Cell 6 — Save to Drive

In [6]:
output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache'
os.makedirs(output_folder, exist_ok=True)
np.save(f'{output_folder}/pred_levit.npy', pred_levit)
print("Saved. Now run the Significance_Tests notebook.")

Saved. Now run the Significance_Tests notebook.


In [7]:
# ============================================================
# Efficiency measurement: parameter count + inference time for LeViT
# Add this as a NEW cell after Cell 6 in SigTest_Part2_LeViT.ipynb
# (reuses the already-loaded model -- no reloading needed)
# ============================================================
import time
import pandas as pd

def measure_inference_time(model, dataset, warmup=3, timed=15):
    it = iter(dataset.repeat())
    for _ in range(warmup):
        batch = next(it)
        _ = model(batch, training=False)
    total_images = 0
    start = time.perf_counter()
    for _ in range(timed):
        batch = next(it)
        _ = model(batch, training=False)
        total_images += batch.shape[0]
    elapsed = time.perf_counter() - start
    ms_per_image = (elapsed / total_images) * 1000
    return ms_per_image

params = levit_model.count_params()
ms_per_img = measure_inference_time(levit_model, test_ds_float32)
print(f"LeViT: {params:,} params, {ms_per_img:.2f} ms/image")

efficiency_df = pd.DataFrame([{"Model": "LeViT", "Params": params, "ms_per_image": ms_per_img}])

output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache'
efficiency_df.to_csv(f'{output_folder}/efficiency_part2.csv', index=False)
print(f"Saved to: {output_folder}/efficiency_part2.csv")

LeViT: 8,591,116 params, 8.76 ms/image
Saved to: /content/drive/MyDrive/thesis_ensemble_outputs/significance_test_cache/efficiency_part2.csv
